# Pip Installs

In [1]:
%pip install azure-ai-ml azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Initial Imports

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from dotenv import load_dotenv


In [5]:
load_dotenv()

True

# Get env Values

In [6]:
subscription_id = os.getenv("SUBSCRIPTION")
resource_group = os.getenv("RESOURCE_GROUP")
workspace_name = os.getenv("WS_NAME")


# Connect to Azure ML workspace

In [7]:
# Handle to the workspace
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

## Get handle to azureml registry for the RAI built in components

In [8]:
ml_client_registry = MLClient(
    credential=credential,
    registry_name="azureml",
    registry_location="eastus",)
print(ml_client_registry)

MLClient(credential=<azure.identity._credentials.default.DefaultAzureCredential object at 0x000002304FEA1390>,
         subscription_id=6c6683e9-e5fe-4038-8519-ce6ebec2ba15,
         resource_group_name=registry-builtin-prod-eastus-01,
         workspace_name=None)


# Define variables needed for RAI.

### This may invove getting the latetst versions of registered models and datasets

## Get path to parent directory

In [9]:
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [10]:
# Import the name of the registered model in Azure ML
from command_job import artifact_path_name

# Import the name of the test and train datasets
from data_migration.migrate import train_data_name, test_data_name

## Get the latest registered train and test datasets

In [11]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input
# Get the latest version of the trained model
latest_data_version = max(
    [int(d.version) for d in ml_client.data.list(name=train_data_name)])

# Get the latest version of the test data
latest_test_data_version = max(
    [int(d.version) for d in ml_client.data.list(name=test_data_name)])

# Define the latest version of the registered model for deployment
train_data = f"azureml:{train_data_name}:{latest_data_version}"

# Define the latest version of the registered model for deployment
test_data = f"azureml:{test_data_name}:{latest_test_data_version}"

In [12]:
print( train_data)
print(test_data)

azureml:admissions-train-data:9
azureml:admissions-test-data:6


## Get latest registered model and set it to model

In [13]:
 # Get the latest version of the trained model
latest_model_version = max(
    [int(m.version) for m in ml_client.models.list(name=artifact_path_name)])

# Define the latest version of the registered model for deployment
model_id = f"{artifact_path_name}:{latest_model_version}" # For Model Info
model = f"azureml:{artifact_path_name}:{latest_model_version}"

In [14]:
print(model_id)

admissions_model:13


In [15]:
print(model)

azureml:admissions_model:13


In [16]:
target_column_name="Accept"

categorical_features = json.dumps(["Gender", "Race_American_Indian_or_Alaska_Native", "Race_Asian", "Race_Black_or_African_American", "Race_Native_Hawaiian_or_Other_Pacific_Islander", "Race_White", "Ethnicity_Hispanic_or_Latino", "LegacyStatus", "FirstGeneration"])

treatment_features = json.dumps(
    [
        "GPA",
        "SAT"
    ]
)


## Helper function to do the submission, which waits for the submitted job to complete:

In [17]:
from azure.ai.ml.entities import PipelineJob
from IPython.core.display import HTML
from IPython.display import display


def submit_and_wait(ml_client, pipeline_job) -> PipelineJob:
    created_job = ml_client.jobs.create_or_update(pipeline_job)
    assert created_job is not None

    print("Pipeline job can be accessed in the following URL:")
    display(HTML('<a href="{0}">{0}</a>'.format(created_job.studio_url)))

    while created_job.status not in [
        "Completed",
        "Failed",
        "Canceled",
        "NotResponding",
    ]:
        time.sleep(30)
        created_job = ml_client.jobs.get(created_job.name)
        print("Latest status : {0}".format(created_job.status))
    assert created_job.status == "Completed"
    return created_job

# Initialize all the RAI components

In [18]:
label="latest"

## Get the Constructor Component

In [19]:
# Initialize the RAI Insights dashboard constructor
rai_constructor_component = ml_client_registry.components.get(name="rai_tabular_insight_constructor", label="latest")


## Get all sub insight components

In [20]:

# Initialize the RAI Causal component to generate causal analysis insights.
rai_causal_component = ml_client_registry.components.get(name="rai_tabular_causal", label="latest")
# Initialize the RAI Counterfactual component to generate counterfactual insights.
rai_counterfactual_component = ml_client_registry.components.get(name="rai_tabular_counterfactual", label="latest")
rai_erroranalysis_component = ml_client_registry.components.get(name="rai_tabular_erroranalysis", label="latest")
rai_explanation_component = ml_client_registry.components.get(name="rai_tabular_explanation", label="latest")


## Get the gather component

In [21]:
rai_gather_component = ml_client_registry.components.get(
    name="rai_tabular_insight_gather", label=label
)

# Define RAI Dashboard Pipeline

## Imports for RAI 

In [22]:
import json
import tempfile
import shutil
from azure.ai.ml import dsl, Input, Output
from azure.ai.ml.constants import AssetTypes

In [23]:

timeout = 10000000


@dsl.pipeline(
    compute="admissions-compute",
    description="Responsible AI Insights pipeline for admissions classification model",
    experiment_name="Responsible_AI_Insights_Admissions_Model",
)
def rai_classification_pipeline(
):
    # Initiate the RAIInsights
    create_rai_job = rai_constructor_component(
        title="RAI Dashboard Example",
        task_type="classification",
        model_info=model_id,
        model_input=Input(type=AssetTypes.MLFLOW_MODEL, path=model),
        train_dataset=Input(type=AssetTypes.MLTABLE, path=train_data),
        test_dataset=Input(type=AssetTypes.MLTABLE, path=test_data),
        target_column_name=target_column_name,
        categorical_column_names=categorical_features,
    )
    create_rai_job.set_limits(timeout=timeout)

    # Add an explanation
    explain_job = rai_explanation_component(
        comment="Explanation for the classification dataset",
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
    )
    explain_job.set_limits(timeout=timeout)

    # Add causal analysis
    causal_job = rai_causal_component(
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
        treatment_features=treatment_features,
    )
    causal_job.set_limits(timeout=timeout)

    # Add error analysis
    erroranalysis_job = rai_erroranalysis_component(
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
    )
    erroranalysis_job.set_limits(timeout=timeout)

    # Combine everything
    rai_gather_job = rai_gather_component(
        constructor=create_rai_job.outputs.rai_insights_dashboard,
        insight_1=explain_job.outputs.explanation,
        insight_2=causal_job.outputs.causal,
        insight_3=erroranalysis_job.outputs.error_analysis,
    )
    rai_gather_job.set_limits(timeout=timeout)


    return {"ux_json": rai_gather_job.outputs.ux_json}


## Creates the pipeline and sets the ux_json as a pipeline level output

In [ ]:
import uuid
unique_code = str(uuid.uuid4())
# Pipeline to construct the RAI Insights
insights_pipeline_job = rai_classification_pipeline()

insights_pipeline_job.outputs.ux_json = Output(
    path=f"azureml://datastores/workspaceblobstore/paths/ux_json_outputs/{unique_code}/",
    mode="upload",
    type="uri_folder",
)


In [ ]:
insights_job = submit_and_wait(ml_client, insights_pipeline_job)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Pipeline job can be accessed in the following URL:


Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Running
Latest status : Completed


# Model Analysis using Responsible AI Insights

## Download ux.json from the RAI job getting the insights

In [25]:
ml_client.jobs.download(
    name="modest_mangos_y5vqmx6hhc",
    output_name="ux_json",
    download_path="./rai_outputs"

)



In [31]:
sub_id = ml_client._operation_scope.subscription_id
rg_name = ml_client._operation_scope.resource_group_name
ws_name = ml_client.workspace_name

expected_uri = f"https://ml.azure.com/model/{model_id}/model_analysis?wsid=/subscriptions/{sub_id}/resourcegroups/{rg_name}/workspaces/{ws_name}"

print(f"Please visit {expected_uri} to see your analysis")

Please visit https://ml.azure.com/model/admissions_model:12/model_analysis?wsid=/subscriptions/a6b2ef6f-dbdd-4051-8a2e-9d1828d81dc2/resourcegroups/rg-admissions/workspaces/Admissions-Workspace to see your analysis
